# Preprocessing Practice


- Tokenize
- Cleansing & Normalization 
- Stemming & Lemmatization 

- Steps
- 1. Find Corpus Dataset (or crawl)
- 2. Preprocessing <br>
    2-1. 의미가 있는 단어단위로 vocabulary  <br>
    2-2. corpus -> tokenize + preprocessing -> sentence ( corpus -> 토큰화 + 전처리 -> 문장)

# 전처리 연습 (Tokenize, Cleansing & Normalization, Stemming & Lemmatization)

1. 데이터셋(corpus)을 찾는다. (만들어진 데이터셋, 크롤링, ...)
    - **🐿️ 이어지는 실습에 쓸 수 있을 법한 데이터 찾기 Tip**
    - 여러 문장으로 이루어진 데이터셋이라면 일단 good
        - 리뷰 등 이어지지 않는 짧은 문장 여러 건 ok
        - 소설 등 이어지는 긴 문장 여러 건 ok
    - 대화형 데이터셋도 good
        - QnA로 구성되어 있는 corpus도 좋고
        - 일반적인 대화로 구성되어 있는 corpus도 좋아요~
    - 특정 도메인에 대한 정보를 담고 있는 데이터셋도 good
2. 전처리를 해본다.
    - 텍스트 자체를 깔끔하게 만드는 것까지만

### 1. Data
citation / 출저: 
https://aihub.or.kr/aihubdata/data/view.do?dataSetSn=625
-  1. Data File 
- Sentences containing expressions and emotions through online interactions sites (comments, etc)
    - Data Folders
        - a. 라벨링데이터
    - labels: text/tag
        - b. 원천데이터
            - content

In [3]:
import json
import pandas as pd
from konlpy.tag import Okt
import glob
import os


### data folder에서 json file 다 로드하기 
- 문제: files too big, won't load

In [2]:
# json_files = glob.glob('data/*.json')

# sentences = []
# okt = Okt()
# all_dfs = []

# for json_filepath in json_files:
#     with open(json_filepath, 'r', encoding='utf-8') as f:
#         json_dict = json.load(f)
#         data = json_dict.get('SJML').get('text')
#         if data:
#             df_temp = pd.DataFrame(data)
#             df_temp['filename'] = os.path.basename(json_filepath)
#             all_dfs.append(df_temp)

# df = pd.concat(all_dfs, ignore_index=True)
# sentences = df['content'].tolist()

# # corpus_text = '.'.join(sentences)
# # print(corpus_text)
# print(df[['content', 'filename']].head(10))
# len(sentences)

##### data from desktop 
- mac os format (local desktop)

In [ ]:
folder_path = os.path.expanduser("~/Desktop/skn_24/practice/NLP_practice_data/031.온라인_구어체_말뭉치_데이터/01.데이터/1.Training_220728_add/원천데이터/TS1")
all_dfs = []

if os.path.exists(folder_path):
    # Get subfolders (e.g., '건강_의학', '스포츠', etc.)
    folders = [n for n in os.listdir(folder_path) if os.path.isdir(os.path.join(folder_path, n))]
    print(f'Total folders found: {len(folders)}')

    for folder in folders: 
        # Recursively find all json files in the subfolder
        json_files = glob.glob(os.path.join(folder_path, folder, '*.json'))
        print(f'Processing Folder: {folder} | Files: {len(json_files)}')

        for file in json_files:
            try:
                with open(file, 'r', encoding='utf-8') as f:
                    json_dict = json.load(f)

                    text_list = json_dict.get('SJML').get('text')
                    
                    if text_list:
                        df_temp = pd.DataFrame(text_list)
                        df_temp['filename'] = os.path.basename(file)
                        df_temp['category'] = folder  
                        all_dfs.append(df_temp)
            except Exception as e:
                print(f"Error reading {file}: {e}")

    # combine 
    if all_dfs:
        df = pd.concat(all_dfs, ignore_index=True)
        sentences = df['content'].tolist()
        
        print(df[['content', 'filename', 'category']].head(10))
        print(f'sentences length: {len(sentences)}')
    else:
        print("No data was found in the JSON files.")

else:
    print(f"Path doesn't exist: {folder_path}")


Total folders found: 13
Processing Folder: 취미 | Files: 1385
Processing Folder: 음식 | Files: 981
Processing Folder: 방송 | Files: 5065
Processing Folder: 과학 | Files: 909
Processing Folder: 인터넷방송 | Files: 6071
Processing Folder: 게임 | Files: 1713
Processing Folder: 엔터테인먼트 | Files: 593
Processing Folder: 일상 | Files: 1203
Processing Folder: 뷰티 | Files: 1373
Processing Folder: 연애_결혼 | Files: 932
Processing Folder: 건강_의학 | Files: 737
Processing Folder: 동물 | Files: 1229
Processing Folder: 유머 | Files: 844
                                             content               filename  \
0   와 다음에 소니에서 무조건 협찬줘야하는 영상이네요. 이 영상보고 너무너무 갖고싶습니다.  BOHB210001410024.json   
1                구독자인데요 kyo님 구매링크 부탁드려도 될까요 렌즈도 포함이용  BOHB210001410024.json   
2                                    랜즈는 머가 적합할까요 ..  BOHB210001410024.json   
3               목소리는 20대 중후반인데 웃는모습은 7살 어린애처럼 순수하시네요  BOHB210001410024.json   
4                사진촬영도 취미로 같이하는데 2천만화소로만 해주지 하

In [ ]:
import os
import glob
import json
import pandas as pd

# 1. Setup paths
folder_path = os.path.expanduser("~/Desktop/skn_24/practice/NLP_practice_data/031.온라인_구어체_말뭉치_데이터/01.데이터/1.Training_220728_add/원천데이터/TS1")
output_dir = "data"

# Create a dedicated folder for the results
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

all_dfs = []    # concat all csv
sentences = []    # concat all csv

if os.path.exists(folder_path):
    folders = [n for n in os.listdir(folder_path) if os.path.isdir(os.path.join(folder_path, n))]
    
    for folder in folders:
        current_folder_dfs = [] # datafame per folder (category)
        json_files = glob.glob(os.path.join(folder_path, folder, '*.json'))
        print(f'Folder: {folder} ,{len(json_files)} files')

        for file in json_files:
            try:
                with open(file, 'r', encoding='utf-8') as f:
                    json_dict = json.load(f)
                    data = json_dict.get('SJML').get('text')
                    
                    if data:
                        df_temp = pd.DataFrame(data)
                        df_temp['filename'] = os.path.basename(file)
                        df_temp['category'] = folder

                        keep_cols = ['content', 'category', 'filename']
                        df_temp = df_temp[[c for c in keep_cols if c in df_temp.columns]]
                        
                        current_folder_dfs.append(df_temp)


            except Exception as e:
                print(f"Not saved {file} error ({e})")

        # save csv of each category df
        if current_folder_dfs:
            category_df = pd.concat(current_folder_dfs, ignore_index=True)
            
            category_file_name = f"{folder}_dataset.csv"
            save_path = os.path.join(output_dir, category_file_name)
            category_df.to_csv(save_path, index=False, encoding='utf-8')
            
            print(f"saved {folder} to {save_path}")
            
            #  다 합친거 
            all_dfs.append(category_df)

# category df 다 합친거 save
if all_dfs:
    final_df = pd.concat(all_dfs, ignore_index=True)
    final_df.to_csv(output_dir+"total_corpus.csv", index=False, encoding='utf-8')
    sentences = final_df['content'].tolist()


Folder: 취미 ,1385 files
saved 취미 to processed_categories/취미_dataset.csv
Folder: 음식 ,981 files
saved 음식 to processed_categories/음식_dataset.csv
Folder: 방송 ,5065 files
saved 방송 to processed_categories/방송_dataset.csv
Folder: 과학 ,909 files
saved 과학 to processed_categories/과학_dataset.csv
Folder: 인터넷방송 ,6071 files
saved 인터넷방송 to processed_categories/인터넷방송_dataset.csv
Folder: 게임 ,1713 files
saved 게임 to processed_categories/게임_dataset.csv
Folder: 엔터테인먼트 ,593 files
saved 엔터테인먼트 to processed_categories/엔터테인먼트_dataset.csv
Folder: 일상 ,1203 files
saved 일상 to processed_categories/일상_dataset.csv
Folder: 뷰티 ,1373 files
saved 뷰티 to processed_categories/뷰티_dataset.csv
Folder: 연애_결혼 ,932 files
saved 연애_결혼 to processed_categories/연애_결혼_dataset.csv
Folder: 건강_의학 ,737 files
saved 건강_의학 to processed_categories/건강_의학_dataset.csv
Folder: 동물 ,1229 fi

In [20]:
all_df = pd.read_csv('total_corpus.csv', encoding='utf-8')
print(all_df.info())
print(all_df.head(10))

<class 'pandas.DataFrame'>
RangeIndex: 5133299 entries, 0 to 5133298
Data columns (total 3 columns):
 #   Column    Dtype
---  ------    -----
 0   content   str  
 1   category  str  
 2   filename  str  
dtypes: str(3)
memory usage: 721.0 MB
None
                                             content category  \
0   와 다음에 소니에서 무조건 협찬줘야하는 영상이네요. 이 영상보고 너무너무 갖고싶습니다.     취미   
1                구독자인데요 kyo님 구매링크 부탁드려도 될까요 렌즈도 포함이용     취미   
2                                    랜즈는 머가 적합할까요 ..     취미   
3               목소리는 20대 중후반인데 웃는모습은 7살 어린애처럼 순수하시네요     취미   
4                사진촬영도 취미로 같이하는데 2천만화소로만 해주지 하는 아쉬움이     취미   
5  안녕하세요 카잘못알인데 혹시 견본 영상은 후보정 하신건가요 카메라를 사게되면 영상편...     취미   
6  NINJA V를 사용 하려고 하는데요. 카메라종류에 따라 HDMI 출력을 30p로 ...     취미   
7                                      아, 뻠프 제대로 받네.     취미   
8                                            갖고 싶네요,     취미   
9  아 이건 정말 너무너무너무너무너무너무 사고싶다 가격, 끄앙 색감이 진짜 너무 이뿌네...     취미   

                filename  
0  BOHB2

In [21]:
all_df['category'].unique()
all_df.columns

Index(['content', 'category', 'filename'], dtype='str')

In [ ]:
# # /Users/kat/desktop/skn_24/practice/NLP_practice_data/031.온라인_구어체_말뭉치_데이터/01.데이터/1.Training_220728_add/원천데이터/TS1/건강_의학

# sentences = []
# okt = Okt()
# all_dfs = []

# # folder_path = os.path.expanduser("~/Desktop/skn_24/practice/NLP_practice_data/031.온라인_구어체_말뭉치_데이터/01.데이터/1.Training_220728_add/원천데이터/TS1/건강_의학")
# folder_path = os.path.expanduser("~/Desktop/skn_24/practice/NLP_practice_data/031.온라인_구어체_말뭉치_데이터/01.데이터/1.Training_220728_add/원천데이터/TS1")


# if os.path.exists(folder_path):
#     folders = [name for name in os.listdir(folder_path) if os.path.isdir(os.path.join(folder_path, name))]
#     print('total folders: ', len(folders))

#     for folder in folders: 
#         json_files = glob.glob(os.path.join(folder_path, folder, '*.json'))
#         print(f'folder: {type(folder)} , num of files: {len(json_files)}')

#         for file in json_files:
#             with open(file, 'r') as f:
#                     json_dict = json.load(f)
#                     data = json_dict.get('SJML').get('text')
#                     if data:
#                         df_temp = pd.DataFrame(data)
#                         df_temp['filename'] = os.path.basename(file)
#                         df_temp['folder'] = folder
#                         all_dfs.append(df_temp)

# else:
#     print(f"File not found: {folder_path}")

# df = pd.concat(all_dfs, ignore_index=True)
# sentences = df['content'].tolist()

# print(df[['content', 'filename']].head(10))
# print(f'len(sentences) : {len(sentences)}')


total folders:  13
folder: <class 'str'> , num of files: 1385
folder: <class 'str'> , num of files: 981
folder: <class 'str'> , num of files: 5065
folder: <class 'str'> , num of files: 909
folder: <class 'str'> , num of files: 6071
folder: <class 'str'> , num of files: 1713
folder: <class 'str'> , num of files: 593
folder: <class 'str'> , num of files: 1203
folder: <class 'str'> , num of files: 1373
folder: <class 'str'> , num of files: 932
folder: <class 'str'> , num of files: 737
folder: <class 'str'> , num of files: 1229
folder: <class 'str'> , num of files: 844
                                             content               filename
0   와 다음에 소니에서 무조건 협찬줘야하는 영상이네요. 이 영상보고 너무너무 갖고싶습니다.  BOHB210001410024.json
1                구독자인데요 kyo님 구매링크 부탁드려도 될까요 렌즈도 포함이용  BOHB210001410024.json
2                                    랜즈는 머가 적합할까요 ..  BOHB210001410024.json
3               목소리는 20대 중후반인데 웃는모습은 7살 어린애처럼 순수하시네요  BOHB210001410024.json
4                사진촬영도 취미로 같이하는데 2천만화소로만 해주지 하는 

### 2. stopwords, stemming, lemmatization
- 불용어(stopwords) 제거
- 어간 추출(Stemming) 및 표제어 추출(Lemmatization)

In [28]:
import re
okt = Okt()

def load_ko_stopwords(filepath): 
    with open(filepath, 'r', encoding='UTF-8') as f:
        return [line.strip() for line in f]

def clean_sentence(sentence, stopwords):
    sentence = re.sub(r'[^가-힣a-zA-Z0-9\s]', '', sentence)  # 한글, 영문, 숫자만 only
    tokens = okt.morphs(sentence, norm=True, stem=True) # tokenize & stem=True==lemmatization & stemming
    tokens = [token for token in tokens if len(token) > 0]
    return [token for token in tokens if token not in stopwords]   # filter stopwords

def clean_sentence_pos(sentence, stopwords):    # pos 적용
    pos_tokens = okt.pos(sentence, norm=True, stem=True)
    pos_filter = {'Noun', 'Verb', 'Adjective', 'Adverb'}
    tokens = [
        token for token, pos in pos_tokens
        if pos in pos_filter and token not in stopwords
    ]
    return tokens 

ko_stopwords = load_ko_stopwords('ko_stopwords.txt')

In [27]:
len(sentences)

5133299

In [5]:
# cleaned_sentences =[]

# for s in sentences:
#     cleaned_tokens = clean_sentence(s, ko_stopwords)
#     cleaned_sentences.append(cleaned_tokens)

# cleaned_sentences

In [29]:
cleaned_sentences = [clean_sentence(s, ko_stopwords) for s in sentences]

# corpus text (one list)
corpus_text = [' '.join(tokens) for tokens in cleaned_sentences]

print(len(corpus_text))

KeyboardInterrupt: 

In [ ]:
display(len(cleaned_sentences))
display(cleaned_sentences)

In [ ]:
corpus_text

### 3. corpus_text json file 저장 
- output is truncated even when opened in text editor

In [ ]:
import json

with open('corpus_text.json', 'w', encoding='utf-8') as f:
    json.dump(corpus_text, f, ensure_ascii=False, indent=2)